# ASL Citizen top-50 — Graph + Spatial + Temporal Transformer

Notebook này tạo subset 50 lớp từ manifest top-200, giữ nguyên official split, tái sử dụng raw pose, tạo graph `[64, 75, 7]`, smoke-test kiến trúc mới và huấn luyện pose-only đầy đủ. Test chỉ được chạy sau khi checkpoint tốt nhất đã được chọn bằng validation macro-F1.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

PROJECT_GIT_URL = 'https://github.com/stillthethrone/silent-signal.git'
PROJECT_GIT_REF = 'feat/videomaev2-rgb-early-stopping'  # @param {type:'string'}
PROJECT_ROOT = Path('/content/silent-signal')
RESULTS_ROOT_STR = '/content/drive/.shortcut-targets-by-id/1-oYEcvJh4ylv_f4AKkJkCjs3FgzjDBFE/silent-signal-results/asl_citizen'  # @param {type:'string'}
RESULTS_ROOT = Path(RESULTS_ROOT_STR)
TOP200_ROOT = RESULTS_ROOT / 'subsets/asl_citizen_asllex_top200'
TOP50_ROOT = RESULTS_ROOT / 'subsets/asl_citizen_asllex_top50'
SOURCE_MANIFEST = TOP200_ROOT / 'manifest.csv'
SOURCE_SELECTION = TOP200_ROOT / 'selection_report.json'
RAW_POSE_ROOT = TOP200_ROOT / 'pose/rtmpose_l_coco_wholebody_384x288/raw'
TOP50_MANIFEST = TOP50_ROOT / 'manifest.csv'
GRAPH_ROOT = TOP50_ROOT / 'graph/asl_citizen_coco_wholebody_v1/t64'
GRAPH_REPORT = TOP50_ROOT / 'reports/graph_preparation_t64.json'
MODEL_ROOT = TOP50_ROOT / 'models/graph_spatial_temporal_transformer_v1'
SMOKE_CHECKPOINT = MODEL_ROOT / 'smoke.pt'
SMOKE_REPORT = TOP50_ROOT / 'reports/graph_transformer_smoke.json'
SEED = 42  # @param {type:'integer'}
TRAINING_ROOT = MODEL_ROOT / f'seed_{SEED}'
RUN_SELECTION = True  # @param {type:'boolean'}
RUN_GRAPH_PREPARATION = True  # @param {type:'boolean'}
RUN_SMOKE = True  # @param {type:'boolean'}
RUN_FULL_TRAINING = True  # @param {type:'boolean'}
OVERWRITE_GRAPH = False  # @param {type:'boolean'}
OVERWRITE_TRAINING = False  # @param {type:'boolean'}
MAX_EPOCHS = 100  # @param {type:'integer'}
BATCH_SIZE = 32  # @param {type:'integer'}
NUM_WORKERS = 0  # @param {type:'integer'}
print('Source top-200:', TOP200_ROOT)
print('Top-50 output:', TOP50_ROOT)

## Chuẩn bị source và môi trường

Đẩy code lên `PROJECT_GIT_REF` trước khi chạy. Colab đã có PyTorch; notebook chỉ cài project ở editable mode.

In [ ]:
import subprocess
import sys

def run(command, *, cwd=None):
    command = [str(item) for item in command]
    print('+', ' '.join(command), flush=True)
    subprocess.run(command, cwd=cwd, check=True)

if not PROJECT_ROOT.exists():
    run(['git', 'clone', '--depth', '1', '--branch', PROJECT_GIT_REF, PROJECT_GIT_URL, PROJECT_ROOT])
elif not (PROJECT_ROOT / '.git').is_dir():
    raise RuntimeError(f'{PROJECT_ROOT} tồn tại nhưng không phải Git repository.')
run([sys.executable, '-m', 'pip', 'install', '--quiet', '-e', PROJECT_ROOT])
run([sys.executable, '-c', 'import torch, silent_signal; print(torch.__version__)'])
PROJECT_COMMIT = subprocess.check_output(
    ['git', 'rev-parse', 'HEAD'], cwd=PROJECT_ROOT, text=True
).strip()
print('Project commit:', PROJECT_COMMIT)

## Chọn 50 lớp và giữ official split

Chỉ các lớp có clip trong cả train, validation và test mới đủ điều kiện. Nhãn được remap về `0..49`; `sample_id`, `signer_id` và `split` không đổi.

In [ ]:
for required in (SOURCE_MANIFEST, SOURCE_SELECTION, RAW_POSE_ROOT):
    if not required.exists():
        raise FileNotFoundError(required)
selection_command = [
    sys.executable, '-u', '-m', 'silent_signal.cli.select_ranked_subset',
    '--manifest', SOURCE_MANIFEST,
    '--selection-report', SOURCE_SELECTION,
    '--output-root', TOP50_ROOT,
    '--classes', 50,
    '--project-commit', PROJECT_COMMIT,
]
if RUN_SELECTION:
    run(selection_command, cwd=PROJECT_ROOT)
elif not TOP50_MANIFEST.is_file():
    raise FileNotFoundError(TOP50_MANIFEST)

In [ ]:
import json
from collections import Counter
from silent_signal.data.manifest import read_manifest

records = read_manifest(TOP50_MANIFEST)
split_counts = Counter(str(record.split) for record in records)
signers = {
    split: {record.signer_id for record in records if record.split == split}
    for split in ('train', 'validation', 'test')
}
assert {record.class_index for record in records} == set(range(50))
assert not (signers['train'] & signers['validation'])
assert not (signers['train'] & signers['test'])
assert not (signers['validation'] & signers['test'])
assert all(
    {record.split for record in records if record.class_index == class_index}
    == {'train', 'validation', 'test'}
    for class_index in range(50)
)
total = len(records)
print('Official split top-50:')
for split in ('train', 'validation', 'test'):
    print(f'- {split}: {split_counts[split]} ({split_counts[split] / total * 100:.2f}%)')
print('Split isolation: PASS')

## Tạo graph cache top-50

Bước này chạy CPU, đọc raw pose top-200 nhưng chỉ xử lý `sample_id` có trong manifest top-50. Có thể dừng và chạy lại với `OVERWRITE_GRAPH=False`.

In [ ]:
graph_command = [
    sys.executable, '-u', '-m', 'silent_signal.cli.prepare_graph',
    '--config', PROJECT_ROOT / 'configs/preprocessing/asl_citizen_top50_graph.yaml',
    '--manifest', TOP50_MANIFEST,
    '--pose-root', RAW_POSE_ROOT,
    '--output-root', GRAPH_ROOT,
    '--report', GRAPH_REPORT,
    '--continue-on-error', '--progress-every', 25,
]
if OVERWRITE_GRAPH:
    graph_command.append('--overwrite')
if RUN_GRAPH_PREPARATION:
    run(graph_command, cwd=PROJECT_ROOT)
elif not GRAPH_REPORT.is_file():
    raise FileNotFoundError(GRAPH_REPORT)
graph_report = json.loads(GRAPH_REPORT.read_text(encoding='utf-8'))
completed = graph_report.get('prepared', 0) + graph_report.get('resumed', 0)
if completed != len(records) or graph_report.get('failed') != 0:
    raise RuntimeError('Graph preparation chưa hoàn tất sạch.')
print('Graph preparation: PASS', completed, '/', len(records))

## Smoke-test Graph + Spatial Transformer + Temporal Transformer

In [ ]:
smoke_command = [
    sys.executable, '-u', '-m', 'silent_signal.cli.check_graph_encoder',
    '--config', PROJECT_ROOT / 'configs/model/asl_citizen_top50_graph_transformer.yaml',
    '--manifest', TOP50_MANIFEST,
    '--graph-root', GRAPH_ROOT,
    '--checkpoint', SMOKE_CHECKPOINT,
    '--report', SMOKE_REPORT,
    '--device', 'cuda',
    '--project-commit', PROJECT_COMMIT,
    '--overwrite',
]
if RUN_SMOKE:
    run(smoke_command, cwd=PROJECT_ROOT)
elif not SMOKE_REPORT.is_file():
    raise FileNotFoundError(SMOKE_REPORT)
smoke = json.loads(SMOKE_REPORT.read_text(encoding='utf-8'))
if smoke.get('state') != 'passed':
    raise RuntimeError('Smoke test chưa PASS.')
print('Transformer smoke test: PASS')

## Full training và held-out test

Trainer dùng class-weighted loss cho train, chọn `best.pt` theo validation macro-F1, lưu `last.pt` để resume và chỉ đánh giá test sau khi chọn xong checkpoint tốt nhất.

In [ ]:
training_report = TRAINING_ROOT / 'evaluation.json'
last_checkpoint = TRAINING_ROOT / 'last.pt'
train_command = [
    sys.executable, '-u', '-m', 'silent_signal.cli.train_graph_transformer',
    '--config', PROJECT_ROOT / 'configs/model/asl_citizen_top50_graph_transformer.yaml',
    '--manifest', TOP50_MANIFEST,
    '--graph-root', GRAPH_ROOT,
    '--output-root', TRAINING_ROOT,
    '--device', 'cuda',
    '--epochs', MAX_EPOCHS,
    '--batch-size', BATCH_SIZE,
    '--num-workers', NUM_WORKERS,
    '--seed', SEED,
    '--project-commit', PROJECT_COMMIT,
]
if OVERWRITE_TRAINING:
    train_command.append('--overwrite')
elif last_checkpoint.is_file() and not training_report.is_file():
    train_command.extend(['--resume', last_checkpoint])
if RUN_FULL_TRAINING and (OVERWRITE_TRAINING or not training_report.is_file()):
    run(train_command, cwd=PROJECT_ROOT)
elif not training_report.is_file():
    raise FileNotFoundError(training_report)
report = json.loads(training_report.read_text(encoding='utf-8'))
print(json.dumps({
    'best_epoch': report['best_epoch'],
    'best_validation_macro_f1': report['best_validation_macro_f1'],
    'test_top1': report['test']['top1'],
    'test_top5': report['test']['top5'],
    'test_macro_f1': report['test']['macro_f1'],
    'split': report['split']['clip_counts'],
}, ensure_ascii=False, indent=2))

## Artifacts trên Drive

- `asl_citizen_asllex_top50/manifest.csv`: manifest 50 lớp giữ official split.
- `selected_50_words.json`: ánh xạ lớp mới sang top-200 và thống kê split thực tế.
- `graph/.../t64/`: graph cache `[64,75,7]`.
- `models/.../seed_42/best.pt`: checkpoint chọn bằng validation.
- `models/.../seed_42/evaluation.json`: kết quả held-out test và confusion matrix.